In [1]:
import os
from pathlib import Path
import math
import pytest
import pandas as pd
import torch
from torch import nn
from torch_geometric.loader import DataLoader

In [2]:
import sys
sys.path.append(str(Path().resolve().parents[1]))
sys.path.append(str(Path().resolve().parents[0]))

In [3]:
from SynRxNet.src.datasets import create_dataset, Splitter, FeatureEngineer

In [4]:
DATA_ROOT = Path("../data")
CLEANED_PATH = DATA_ROOT / "processed" / "cleaned_drugcomb.csv"
CCLE_PCA_PATH = DATA_ROOT / "processed" / "ccle_cell_line_pca_100.csv"
print(DATA_ROOT,Path.exists(CLEANED_PATH), Path.exists(CCLE_PCA_PATH))

../data True True


## Basic Sanity Checks

In [5]:
@pytest.mark.skipif(not CLEANED_PATH.exists(), reason="Cleaned data file not found")
def test_cleaned_dataset_structure():
    df = pd.read_csv(CLEANED_PATH)
    required = {"ID", "Drug1", "Drug2", "Cell line", "ZIP", "smiles_drug1", "smiles_drug2"}
    missing = required - set(df.columns)
    assert not missing, f"Missing columns in cleaned dataset: {missing}"
    assert len(df) > 0, "Cleaned dataset is empty"
    print("Cleaned dataset structure is valid.")

In [6]:
test_cleaned_dataset_structure()

Cleaned dataset structure is valid.


## Splitter Sanity

In [19]:
@pytest.mark.skipif(not CLEANED_PATH.exists(), reason="Cleaned data file not found")
@pytest.mark.parametrize("strategy", ["random", "lodo", "loco"])
def test_splitter_strategies(strategy):
    df = pd.read_csv(CLEANED_PATH)
    df = df.sample(min(200, len(df)), random_state=42).reset_index(drop=True)

    splitter = Splitter(strategy=strategy, seed=123)
    wut = splitter.split(df)
    train, val, test = wut

    # Basic Checks
    assert len(train) + len(val) + len(test) == len(df), "Data split sizes do not add up"
    assert len(set(train['ID']) & set(val['ID'])) == 0, "Train and Val sets overlap"
    assert len(set(train['ID']) & set(test['ID'])) == 0, "Train and Test sets overlap"
    assert len(set(val['ID']) & set(test['ID'])) == 0, "Val and Test sets overlap"
    print(f"{strategy} split: Train={len(train)}, Val={len(val)}, Test={len(test)}")

    if strategy == "lodo":
        # TODO: Implement LODO specific checks
        pass
    if strategy == "loco":
        train_cell_lines = set(train['Cell line'])
        test_cell_lines = set(test['Cell line'])
        assert train_cell_lines.isdisjoint(test_cell_lines), "Train and Test cell lines overlap in LOCO split"
    print(f"{strategy} splitter passed all tests.")

In [20]:
test_splitter_strategies("random")
test_splitter_strategies("loco")
# test_splitter_strategies("lodo")

random split: Train=160, Val=20, Test=20
random splitter passed all tests.
loco split: Train=168, Val=22, Test=10
loco splitter passed all tests.


In [21]:
test_splitter_strategies("lodo")

{np.str_('Bleomycin sulfate'), np.str_('Chloroquine'), np.str_('GSK-2879552'), np.str_('BMS-509744'), np.str_('Artesunate'), np.str_('AZACYTIDINE'), np.str_('LEVAMISOLE'), np.str_('Bardoxolone methyl'), np.str_('BLEOMYCIN'), np.str_('STF-62247'), np.str_('34793-34-5'), np.str_('3-AMINO-2-OXAZOLIDINONE'), np.str_('Marizomib / Salinosporamide A'), np.str_('Nisoxetine'), np.str_('Everolimus'), np.str_('J113397'), np.str_('Decoquinate')}
lodo split: Train=142, Val=18, Test=40
lodo splitter passed all tests.


## FeatureEngineer Basic Operations

In [9]:
def test_feature_engineer_smiles_and_descriptors():
    fe = FeatureEngineer(cache_dir=DATA_ROOT / "cache_test")
    smiles_valid = "CCO"
    smiles_invalid = "not_a_smiles"

    assert fe.validate_smiles(smiles_valid) is True
    assert fe.validate_smiles(smiles_invalid) is False

    emb = fe.encode_smiles(smiles_valid)
    assert isinstance(emb, torch.Tensor)
    assert emb.ndim == 1 and emb.shape[0] == 768

    desc = fe.compute_descriptors(smiles_valid)
    assert isinstance(desc, torch.Tensor)
    assert desc.ndim == 1
    assert desc.shape[0] == len(fe.desc_names)

    # 3D features
    coords = fe.compute_3d_features(smiles_valid)
    assert isinstance(coords, torch.Tensor)
    assert coords.ndim == 1  # flattened distance vector
    print(coords.shape)
    assert coords.shape[0] == 1000
    print("Feature engineering tests passed.")

In [10]:
test_feature_engineer_smiles_and_descriptors()

torch.Size([1000])
Feature engineering tests passed.


[08:24:54] SMILES Parse Error: syntax error while parsing: not_a_smiles
[08:24:54] SMILES Parse Error: check for mistakes around position 3:
[08:24:54] not_a_smiles
[08:24:54] ~~^
[08:24:54] SMILES Parse Error: Failed parsing SMILES 'not_a_smiles' for input: 'not_a_smiles'


## Dataset + PyG Dataloader integration

In [5]:
def _make_small_dataset(dataset_type = "graph", subset = "trian", strategy="random"):

    kwargs = {}
    if CCLE_PCA_PATH.exists():
        kwargs['cell_line_features_path'] = str(CCLE_PCA_PATH)
    
    dataset = create_dataset(
        csv_path=str(CLEANED_PATH),
        dataset_type=dataset_type,
        split_strategy=strategy,
        split_seed=42,
        cache_dir=str(DATA_ROOT / "cache_test"),
        **kwargs
    )
    return dataset

In [6]:
@pytest.mark.skipif(not CLEANED_PATH.exists(), reason="Cleaned data file not found")
def test_graph_dataset_and_loader_basic():
    ds = _make_small_dataset(dataset_type="graph", subset="train", strategy="random")
    assert len(ds) > 0, "Graph dataset is empty"

    item = ds[0]

    for k in ["drug1_graph", "drug2_graph", "cell_line", "targets", "metadata"]:
        assert k in item

    g1 = item['drug1_graph']
    g2 = item['drug2_graph']

    assert hasattr(g1, 'x') and hasattr(g1, 'edge_index'), "Drug1 graph missing attributes"
    assert hasattr(g2, 'x') and hasattr(g2, 'edge_index'), "Drug2 graph missing attributes"

    loader = DataLoader(ds, batch_size=8, shuffle=True)
    batch = next(iter(loader))

    assert hasattr(batch, 'drug1_graph') and hasattr(batch, 'drug2_graph'), "Batch missing graph attributes"
    assert batch.drug1_graph.num_graphs == 8, "Batch size mismatch for drug1_graph"
    assert batch.drug2_graph.num_graphs == 8, "Batch size mismatch for drug2_graph"
    print("Graph dataset and DataLoader tests passed.")

In [7]:
test_graph_dataset_and_loader_basic()

{'A2058_SKIN': tensor([ 2.3614e+00, -8.0473e-01,  1.2114e+00,  2.7458e-02, -7.2674e-01,
         4.2603e-01, -4.9647e-01, -1.9933e+00,  2.8774e-01, -6.1618e-01,
         2.3730e+00,  7.2281e-02, -1.4651e+00, -6.4087e-01, -7.6468e-01,
        -5.7516e-01, -4.0507e-01,  5.9002e-01, -1.1988e-01, -7.5867e-01,
         8.5449e-02, -9.4932e-01,  3.6563e-01, -1.7752e-01,  1.6140e+00,
         6.9930e-03, -5.3283e-01,  1.1902e+00, -4.5640e-01, -5.4237e-01,
        -2.0248e-02,  1.9409e-01,  2.5728e+00,  3.5694e-02,  1.3633e+00,
        -1.9078e-01,  1.0380e+00, -7.6068e-01,  1.6561e+00,  1.2139e+00,
         4.3014e-01, -3.7819e-17]), 'A2780_OVARY': tensor([ 2.1442e+00, -9.1185e-01,  9.7690e-01,  2.3825e-02,  8.6148e-01,
         5.7660e-01, -2.1389e-02,  4.5191e-01,  9.1306e-03, -2.4835e-01,
         4.9157e-01,  9.7478e-02, -1.1570e+00, -8.9081e-01, -6.8882e-01,
        -8.6180e-01,  2.5605e-01, -1.1481e-01,  1.2053e+00, -4.7322e-01,
         2.6433e-01, -3.3828e-01, -3.8902e-01, -1.2174e-01

KeyError: 'JHH-136'